Imports

In [1]:
import torch
import pandas as pd

from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification

Get the data

In [4]:
df = pd.read_csv("final_df_fixed.csv")

df.head()

,Unnamed: 0,title,artist,disorder,sentiment_direction,sentiment_score,lyric,emotion_anger_score,emotion_disgust_score,emotion_fear_score,emotion_joy_score,emotion_neutral_score,emotion_sadness_score,emotion_surprise_score,anonymized_author_id,anonymized_id,description,mean_arousal_VAD,mean_dominance_VAD,mean_valence_VAD
0,0,Burnin Bridges / Long Day (feat. IDK),Quadeca,depression,POSITIVE,0.9971,Highest To Lowest: Quadeca LyricsQuadeca's Son...,0.0294,0.0014,0.0132,0.2346,0.0993,0.0832,0.5390,0ef1ff6f271d5cb3541f6995,0ef1ff6f271d5cb3541f6995,"18. He/They. Bisexual. Socialist. Filmmaker, m...",0.060463,0.075335,0.055654
1,1,She's A Lady,Tom Jones,control,POSITIVE,0.9988,She’s a Lady Lyrics[Verse 1]\nWell she's all y...,0.0271,0.0203,0.0071,0.1162,0.1262,0.0335,0.6696,12b35a1f4485cf4aff1a634a,12b35a1f4485cf4aff1a634a,One song everyday (hopefully) of 2020.,0.249911,0.140663,0.105500
2,2,Lilies of the Valley,David Byrne,control,POSITIVE,0.9532,Lilies of the Valley Lyrics[Verse 1]\nMomma sh...,0.0252,0.0054,0.0556,0.5990,0.0678,0.2116,0.0354,12b35a1f4485cf4aff1a634a,12b35a1f4485cf4aff1a634a,One song everyday (hopefully) of 2020.,0.225513,0.093785,-0.189674
3,3,School's Out,Alice Cooper,control,NEGATIVE,0.9995,"School’s Out Lyrics[Verse 1]\nWell, we got no ...",0.1258,0.1330,0.0149,0.0306,0.1615,0.5014,0.0330,12b35a1f4485cf4aff1a634a,12b35a1f4485cf4aff1a634a,One song everyday (hopefully) of 2020.,0.199403,0.061629,-0.165449
4,4,Call My Friends,Shawn Mendes,depression,POSITIVE,0.9978,"Call My Friends Lyrics[Verse 1]\nRight now, I'...",0.0016,0.0006,0.0015,0.0075,0.0055,0.9777,0.0057,3e06169cafdc66d26a9b9ebb,3e06169cafdc66d26a9b9ebb,ariana & shawn + tom | he/him | 19,0.171640,0.082034,0.017132


In [2]:
MODEL_NAME = "RobroKools/vad-bert"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.to(device)

model.eval()

print("Using device:", device)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: RobroKools/vad-bert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Using device: cpu


In [6]:
TEXT_COLUMN = "description"

df[TEXT_COLUMN] = (
    df[TEXT_COLUMN]
    .fillna("")
    .astype(str)
    .str.strip()
)

In [7]:
import re

def prepare_tweet(text):
    text = str(text)

    text = re.sub(r"https?://\S+|www\.\S+", " URL ", text)
    text = re.sub(r"@\w+", " USER ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [8]:
df["tweet_model_text"] = df[TEXT_COLUMN].apply(prepare_tweet)

Run in batches

In [9]:
def predict_vad(
        texts, 
        tokenizer,
        model,
        device,
        batch_size=32,
        max_length=256
):
    predictions = []

    texts = list(texts)

    for start in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[start:start + batch_size]

        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        encoded = {
            key: value.to(device) for key, value in encoded.items()
        }

        with torch.inference_mode():
            outputs = model(**encoded)
            batch_predictions = outputs.logits.cpu().numpy()

        predictions.extend(batch_predictions)

    return predictions

Sanity check

In [13]:
test_sentences = [
    "I won the lottery today and I have never been happier in my life.",
    "I am crying because everything feels hopeless and meaningless.",
    "I am furious and I cannot believe this happened!",
    "I am sitting quietly and reading a book.",
    "The meeting is scheduled for tomorrow afternoon."
]

test_predictions = predict_vad(
    texts=test_sentences,
    tokenizer=tokenizer,
    model=model,
    device=device,
    batch_size=5,
)

test_results = pd.DataFrame(
    test_predictions,
    columns=["valence", "arousal", "dominance"])

test_results.insert(0, "text", test_sentences)

test_results

  0%|          | 0/1 [00:00<?, ?it/s]

,text,valence,arousal,dominance
0,I won the lottery today and I have never been ...,3.638910,3.552541,3.430506
1,I am crying because everything feels hopeless ...,1.852431,3.476166,2.449579
2,I am furious and I cannot believe this happened!,2.273189,4.348909,2.899585
3,I am sitting quietly and reading a book.,3.102197,2.736310,2.835338
4,The meeting is scheduled for tomorrow afternoon.,3.125237,3.108350,3.363207


In [11]:
vad_predictions = predict_vad(
    texts=df["tweet_model_text"],
    tokenizer=tokenizer,
    model=model,
    device=device,
    batch_size=32,
    max_length=256
)

  0%|          | 0/1743 [00:00<?, ?it/s]

KeyboardInterrupt: 

VAD to df

In [ ]:
vad_df = pd.DataFrame(
    vad_predictions,
    columns=["tweet_valence",
             "tweet_arousal",
             "tweet_dominance"],
    index = df.index
)

df = pd.concat([df, vad_df], axis=1)

In [ ]:
df[
    [
        TEXT_COLUMN,
        "tweet_valence",
        "tweet_arousal",
        "tweet_dominance"
    ]
].head(10)

Save

In [ ]:
df.to_csv("tweet_lyrics_vad.csv", index=False)

To try

In [ ]:
# df.to_parquet("tweet_lyrics_vad.parquet", index=False)